In [ ]:
import requests
import pandas as pd

url = 'https://archive-api.open-meteo.com/v1/archive'
params = {
    'latitude': 41.0082,
    'longitude': 28.9784,
    'start_date': '2023-01-01',
    'end_date': '2024-12-31',
    'daily': 'temperature_2m_mean,temperature_2m_min,temperature_2m_max,precipitation_sum,windspeed_10m_max,weathercode',
    'format': 'csv'
}

response = requests.get(url, params=params)
with open('../data/istanbul_weather.csv', 'w', encoding='utf-8') as f:
    f.write(response.text)

print('done')
pd.read_csv('../data/istanbul_weather.csv', skiprows=3, encoding='utf-8').head()

In [ ]:
import requests
import pandas as pd
import os

resp = requests.get('https://data.ibb.gov.tr/en/api/3/action/package_show',
                    params={'id': 'hourly-traffic-density-data-set'})
resources = resp.json()['result']['resources']

os.makedirs('../data/traffic_raw', exist_ok=True)

for r in resources:
    name = r['name']
    url = r['url']

    if '2023' not in name and '2024' not in name:
        continue

    filename = url.split('/')[-1]
    filepath = f'../data/traffic_raw/{filename}'

    if os.path.exists(filepath):
        print(f'{name}: already downloaded')
        continue

    print(f'downloading {name}...')
    response = requests.get(url)
    with open(filepath, 'wb') as f:
        f.write(response.content)
    print(f'{name}: done')

print('all done')

In [ ]:
import pandas as pd
import glob

files = glob.glob("../data/traffic_raw/*.csv")
all_data = []

for f in sorted(files):
    df = pd.read_csv(f)
    all_data.append(df)
    print(f.split("/")[-1], len(df), "rows")

traffic = pd.concat(all_data, ignore_index=True)
traffic["DATE_TIME"] = pd.to_datetime(traffic["DATE_TIME"])
traffic["date"] = traffic["DATE_TIME"].dt.date

daily_speed = traffic.groupby("date")["AVERAGE_SPEED"].mean()
daily_vehicles = traffic.groupby("date")["NUMBER_OF_VEHICLES"].sum()

daily = daily_speed.reset_index()
daily.columns = ["date", "avg_speed"]
daily["total_vehicles"] = daily_vehicles.values

daily.to_csv("../data/istanbul_traffic_daily.csv", index=False)
print("saved", len(daily), "days")
daily.head()
